# Module 10 — Phân tích nhân tố, phân tích cụm & giới thiệu SEM/đa cấp
Notebook RIÊNG cho Module 10 — tái lập EFA (2 nhân tố) trên dữ liệu lớp học, và K-means + chỉ số Silhouette trên dữ liệu PISA 195 trường thật. Cần cài `factor_analyzer` và `scikit-learn` (đã có sẵn trên Colab).

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

## 1. Phân tích nhân tố khám phá (EFA)
🎯 **Phương pháp này trả lời câu hỏi gì?** EFA trả lời: 6 câu hỏi quan sát được (h1-h3, a1-a3) có thể rút gọn thành một số ÍT khái niệm ẩn (latent factor) hay không, và câu hỏi nào "thuộc về" khái niệm ẩn nào?

In [ ]:
items = df[["h1","h2","h3","a1","a2","a3"]].dropna()
print("n =", len(items))

# Tính Bartlett & KMO thủ công (tránh lỗi tương thích numpy phiên bản mới trong factor_analyzer)
R = items.corr().values.copy()
n_obs, p_vars = items.shape
det_R = np.linalg.det(R)
chi2 = -(n_obs - 1 - (2*p_vars + 5)/6) * np.log(det_R)
dof = p_vars*(p_vars-1)/2
p_val = 1 - stats.chi2.cdf(chi2, dof)

R_inv = np.linalg.inv(R)
D = np.diag(1/np.sqrt(np.diag(R_inv)))
partial_corr = -D @ R_inv @ D
np.fill_diagonal(partial_corr, 0)
R2 = R.copy(); np.fill_diagonal(R2, 0)
kmo_model = np.sum(R2**2) / (np.sum(R2**2) + np.sum(partial_corr**2))

print(f"Bartlett: chi2={chi2:.2f}, df={dof:.0f}, p={p_val:.2e}")
print(f"KMO = {kmo_model:.3f}")

#### 📤 Đầu ra thật
KMO=0,688 (mức chấp nhận được, >0,6). Bartlett p<0,001 (có ý nghĩa) → dữ liệu đủ điều kiện chạy EFA.

In [ ]:
from factor_analyzer import FactorAnalyzer
import pandas as pd

fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items)
loadings = pd.DataFrame(fa.loadings_, index=items.columns, columns=["Nhân tố 1", "Nhân tố 2"])
print(loadings.round(3))

ev, _ = fa.get_eigenvalues()
print("\nEigenvalues:", ev[:4].round(3))

#### 📤 Đầu ra thật
a1-a3 tải rõ lên Nhân tố 1 (0,70-0,73); h1-h3 tải rõ lên Nhân tố 2 (0,60-0,75). Eigenvalue: NT1=2,21, NT2=1,73 (cả 2 đều >1). ✅ EFA "phát hiện lại" đúng 2 khái niệm ẩn (hứng thú, lo âu) đã dùng để tạo ra dữ liệu — xác nhận phương pháp hoạt động đúng.

## 2. Phân tích cụm K-means — dữ liệu PISA thật (195 trường)
🎯 **Phương pháp này trả lời câu hỏi gì?** K-means trả lời: 195 trường có thể nhóm thành một số cụm có HỒ SƠ (profile) tương tự nhau trên 5 chỉ số WLE hay không — khác EFA ở việc nhóm ĐỐI TƯỢNG (trường), không phải nhóm BIẾN.

In [ ]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

vars_ = ["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP"]
d = truong[vars_].dropna()
X = StandardScaler().fit_transform(d)
print("n =", len(d))

for k in [2,3,4]:
    km = KMeans(n_clusters=k, random_state=2026, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    print(f"k={k}: silhouette={sil:.3f}, cỡ cụm={list(np.bincount(km.labels_))}")

#### 📤 Đầu ra thật
k=2 có Silhouette cao nhất (0,252) — tối ưu thống kê thuần tuý. k=3 (0,180) có Silhouette thấp hơn nhưng cho câu chuyện diễn giải rõ hơn — đánh đổi kinh điển giữa tối ưu thống kê và khả năng diễn giải.

In [ ]:
km3 = KMeans(n_clusters=3, random_state=2026, n_init=10).fit(X)
d2 = d.copy(); d2["cluster"] = km3.labels_
print(d2.groupby("cluster")[vars_].mean().round(3))
print("\nCỡ mỗi cụm:")
print(d2["cluster"].value_counts().sort_index())

#### 📤 Đầu ra thật
Cụm 1 (n=49) có STAFFSHORT=+1,33, EDUSHORT=+1,51 (đều dương cao) → hồ sơ "khó khăn chồng chất": thiếu nhân sự VÀ học liệu cùng lúc. Cụm 0 (n=64) có EDULEAD=+1,42, DIGPREP=+1,22 → hồ sơ "lãnh đạo tốt, sẵn sàng số hoá cao".

### ⚠️ K-means nhạy với điểm khởi tạo — PSPP cho cỡ cụm KHÁC Python
Chạy `QUICK CLUSTER` trong PSPP với cùng dữ liệu, cùng k=3, cho cỡ cụm 37/128/30 — khác hẳn 64/49/82 của Python. Đây KHÔNG phải lỗi, mà là đặc tính đã biết của K-means: thuật toán rất nhạy với điểm khởi tạo (initial centroids), và 2 phần mềm dùng chiến lược khởi tạo khác nhau.

In [ ]:
# Minh hoạ: chạy K-means nhiều lần với các khởi tạo ngẫu nhiên khác nhau (không cố định random_state)
results = []
for trial in range(5):
    km_trial = KMeans(n_clusters=3, n_init=1, random_state=trial).fit(X)
    results.append(sorted(np.bincount(km_trial.labels_)))
print("Cỡ cụm (đã sắp xếp) qua 5 lần chạy với n_init=1 (chỉ 1 lần khởi tạo mỗi lần):")
for r in results:
    print(" ", r)
print("\n-> Kết quả DAO ĐỘNG giữa các lần chạy khi chỉ khởi tạo 1 lần (n_init=1).")
print("Dùng n_init=10 (mặc định khuyến nghị) để lấy kết quả ổn định nhất trong 10 lần thử.")

## 3. Chỉ số Silhouette — công thức và trực giác
🎯 **Phương pháp này trả lời câu hỏi gì?** Silhouette đo CHẤT LƯỢNG phân cụm cho từng điểm dữ liệu: điểm đó có nằm "gọn" trong cụm của mình (gần các điểm cùng cụm, xa cụm khác) hay không.

In [ ]:
from sklearn.metrics import silhouette_samples
sil_vals = silhouette_samples(X, km3.labels_)
d3 = d.copy(); d3["cluster"] = km3.labels_; d3["silhouette"] = sil_vals
print("Silhouette trung bình mỗi cụm:")
print(d3.groupby("cluster")["silhouette"].mean().round(3))
print(f"\nSố trường có silhouette ÂM (có thể bị gán nhầm cụm): {(sil_vals<0).sum()} / {len(sil_vals)}")

#### 📤 Đầu ra thật
Một số trường có Silhouette âm — dấu hiệu ranh giới giữa các cụm không hoàn toàn rõ ràng (khớp với Silhouette tổng thể chỉ ở mức 0,18, không cao). Đây là lý do quan trọng để BÁO CÁO TRUNG THỰC chất lượng phân cụm, không chỉ trình bày kết quả như thể các cụm tách biệt hoàn hảo.